# ETL Demo Pipeline

This notebook performs ETL on the demo_sales table and creates an etl_demo_output table.

In [ ]:
# Widget setup for catalog and schema (safe defaults)
dbutils.widgets.text("catalog", "hive_metastore")
dbutils.widgets.text("schema", "default")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
print(f"➡️ Using catalog={catalog}, schema={schema}")

In [ ]:
import os
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

catalog = os.getenv("DATABRICKS_BUNDLE_VAR_catalog", catalog)
schema = os.getenv("DATABRICKS_BUNDLE_VAR_schema", schema)
if not catalog:
    raise ValueError("❌ No catalog provided. Check databricks.yml target overrides.")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE {schema}")

input_table = f"{catalog}.{schema}.demo_sales"
output_table = f"{catalog}.{schema}.etl_demo_output"

print(f"Reading input: {input_table}")
df = spark.table(input_table)

df_out = df.withColumn("amount_with_discount", df.amount * 0.9)

print(f"Writing output: {output_table}")
df_out.write.mode("overwrite").saveAsTable(output_table)

print(f"✅ ETL complete, table {output_table} created.")

In [ ]:
display(spark.table(output_table))